In [1]:
from pathlib import Path
import duckdb
from sklearn.model_selection import StratifiedKFold
import pandas as pd
from itertools import product
import os
import sys
sys.path.append(str(Path(os.getcwd()).parent.absolute()))

from ringer_zero.utils import walk_paths

import colorlog
logger = colorlog.getLogger()

In [2]:
BASE_QEUERY = """
SELECT 
    id,
    target,
    el_lhmedium,
    el_lhvloose,
    IF({et_col} < 0, 0, {et_col}) AS {et_col}_normalized,
    ABS({eta_col}) AS {eta_col}_normalized,
    CASE WHEN el_lhmedium AND target=1 THEN TRUE
         WHEN NOT el_lhvloose AND target=0 THEN FALSE
         ELSE NULL END AS standard_binning_label,
FROM read_parquet('{table_data_path}')
WHERE {et_col}_normalized >= {et_bin_left} AND
      {et_col}_normalized < {et_bin_right} AND
      {eta_col}_normalized >= {eta_bin_left} AND
      {eta_col}_normalized < {eta_bin_right} AND
      standard_binning_label IS NOT NULL;
"""

RESULT_QUERY = """
SELECT
    id,
    CASE WHEN el_lhmedium AND target=1 THEN TRUE
         WHEN NOT el_lhvloose AND target=0 THEN FALSE
         ELSE NULL END AS standard_binning_label,
    NULL AS standard_binning_kfold
FROM read_parquet('{table_data_path}');
"""

In [ ]:
dataset_dir = Path('/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction')
et_bins = [15, 20, 30, 40, 50, int(1e12)]
et_col = 'trig_L2_calo_et'
eta_bins = [0, 0.8, 1.37, 1.52, 2.500001]
eta_col = 'trig_L2_calo_eta'
splitter = StratifiedKFold(n_splits=5, shuffle=True)

In [4]:
result_query = RESULT_QUERY.format(
    table_data_path=str(dataset_dir / 'data.parquet' / 'data_*.parquet')
)
results_df = duckdb.query(result_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
results_df

,id,standard_binning_label,standard_binning_kfold
0,0,False,<NA>
1,1,False,<NA>
2,2,False,<NA>
3,3,False,<NA>
4,4,False,<NA>
...,...,...,...
7247094,7247094,True,<NA>
7247095,7247095,True,<NA>
7247096,7247096,True,<NA>
7247097,7247097,True,<NA>


In [5]:
results_df.dtypes

id                        uint64[pyarrow]
standard_binning_label      bool[pyarrow]
standard_binning_kfold     int32[pyarrow]
dtype: object

In [6]:
bins_iterator = product(zip(et_bins[:-1], et_bins[1:]), zip(eta_bins[:-1], eta_bins[1:]))
for (et_bin_left, et_bin_right), (eta_bin_left, eta_bin_right) in bins_iterator:
    print(f'Processing bin: ET in [{et_bin_left}, {et_bin_right}), ETA in [{eta_bin_left}, {eta_bin_right})')
    current_query = BASE_QEUERY.format(
        et_col=et_col,
        eta_col=eta_col,
        table_data_path=str(dataset_dir / 'data.parquet' / 'data_*.parquet'),
        et_bin_left=int(et_bin_left*1e3),
        et_bin_right=int(et_bin_right*1e3),
        eta_bin_left=eta_bin_left,
        eta_bin_right=eta_bin_right
    )
    logger.debug(f'Executing query:\n{current_query}')
    df = duckdb.query(current_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
    id_col = df['id'].astype(int)
    label_col = df['standard_binning_label'].astype(int)
    for fold, (_, val_idx) in enumerate(splitter.split(id_col, label_col)):
        print(f'Assigning fold {fold} to {len(val_idx)} samples')
        results_df.loc[results_df['id'].isin(df['id'].iloc[val_idx]), 'standard_binning_kfold'] = fold

Processing bin: ET in [15, 20), ETA in [0, 0.8)
Assigning fold 0 to 44776 samples
Assigning fold 1 to 44776 samples
Assigning fold 2 to 44776 samples
Assigning fold 3 to 44776 samples
Assigning fold 4 to 44775 samples
Processing bin: ET in [15, 20), ETA in [0.8, 1.37)
Assigning fold 0 to 34479 samples
Assigning fold 1 to 34479 samples
Assigning fold 2 to 34479 samples
Assigning fold 3 to 34478 samples
Assigning fold 4 to 34478 samples
Processing bin: ET in [15, 20), ETA in [1.37, 1.52)
Assigning fold 0 to 8044 samples
Assigning fold 1 to 8044 samples
Assigning fold 2 to 8044 samples
Assigning fold 3 to 8044 samples
Assigning fold 4 to 8044 samples
Processing bin: ET in [15, 20), ETA in [1.52, 2.500001)
Assigning fold 0 to 49634 samples
Assigning fold 1 to 49633 samples
Assigning fold 2 to 49633 samples
Assigning fold 3 to 49633 samples
Assigning fold 4 to 49633 samples
Processing bin: ET in [20, 30), ETA in [0, 0.8)
Assigning fold 0 to 47248 samples
Assigning fold 1 to 47248 samples
As

In [7]:
results_df['standard_binning_kfold'] = results_df['standard_binning_kfold'].astype('uint8[pyarrow]')
results_df

,id,standard_binning_label,standard_binning_kfold
0,0,False,2
1,1,False,3
2,2,False,4
3,3,False,<NA>
4,4,False,<NA>
...,...,...,...
7247094,7247094,True,0
7247095,7247095,True,2
7247096,7247096,True,2
7247097,7247097,True,4


In [8]:
results_df.dtypes

id                        uint64[pyarrow]
standard_binning_label      bool[pyarrow]
standard_binning_kfold     uint8[pyarrow]
dtype: object

# Validate

## Check KFold

In [9]:
results_df.value_counts('standard_binning_kfold', dropna=False)

standard_binning_kfold
<NA>    4067354
0        635956
1        635953
2        635948
3        635946
4        635942
Name: count, dtype: int64

## Check Label

In [10]:
results_df.value_counts('standard_binning_label', dropna=False)

standard_binning_label
False    4918564
True     1922802
<NA>      405733
Name: count, dtype: int64

# Saving data

In [11]:
sample_per_batch = 100_000
output_dir = dataset_dir / 'standard_binning_kfold.parquet'
output_dir.mkdir(exist_ok=True)
for file_count, start in enumerate(range(0, len(results_df), sample_per_batch)):
    end = start + sample_per_batch
    if end > len(results_df):
        end = len(results_df)
    print(f'Saving batch {file_count} with samples from index {start} to {end}')
    batch_df = results_df.iloc[start:end]
    batch_path = output_dir / f'standard_binning_kfold_{file_count:04d}.parquet'
    batch_df.to_parquet(batch_path, index=False)

Saving batch 0 with samples from index 0 to 100000
Saving batch 1 with samples from index 100000 to 200000
Saving batch 2 with samples from index 200000 to 300000
Saving batch 3 with samples from index 300000 to 400000
Saving batch 4 with samples from index 400000 to 500000
Saving batch 5 with samples from index 500000 to 600000
Saving batch 6 with samples from index 600000 to 700000
Saving batch 7 with samples from index 700000 to 800000
Saving batch 8 with samples from index 800000 to 900000
Saving batch 9 with samples from index 900000 to 1000000
Saving batch 10 with samples from index 1000000 to 1100000
Saving batch 11 with samples from index 1100000 to 1200000
Saving batch 12 with samples from index 1200000 to 1300000
Saving batch 13 with samples from index 1300000 to 1400000
Saving batch 14 with samples from index 1400000 to 1500000
Saving batch 15 with samples from index 1500000 to 1600000
Saving batch 16 with samples from index 1600000 to 1700000
Saving batch 17 with samples fr